In [3]:
import os
import numpy as np
from dataclasses import dataclass
from pathlib import Path

In [49]:
@dataclass
class CreateEmbendingsConfig:
    root_dir : Path 
    source_path : str 
    faiss_index : str
    provider: str
    model: str
    base_url : str
    temperature: float
    top_p : float
    max_tokens: int
    timeout : int 
    embedding_model: str
    load_faiss_index: str
    documents_path: str
    top_k : int
    system_prompt  : str

In [15]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\src')

In [16]:
from ServicenowGamification.Utils.Common import *
from ServicenowGamification.Constant import * 
from ServicenowGamification.Entity.Config_Entity import * 

In [17]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification')

In [50]:
class ConfigurationManager:
    def __init__(self,
                 config_path = CONFIG_FILE_PATH,
                 params_path =PARAMS_FILE_PATH
                 ):
        
        self.config=Read_Yaml(Path(config_path))
        self.params=Read_Yaml(Path(params_path))

        self.Create_Embeddings=self.config.Create_Embeddings
    
    def get_create_embendings_config(self) -> CreateEmbendingsConfig:
        Config=self.Create_Embeddings
        create_directories([Config.root_dir])
        
        Create_Embeddings_Config=CreateEmbendingsConfig(
            root_dir=Config.root_dir,
            source_path=Config.source_path,
            faiss_index=Config.faiss_index,
            provider=Config.provider,
            model=Config.model, 
            base_url=Config.base_url, 
            temperature =Config.temperature,
            top_p =Config.top_p,
            max_tokens =Config.max_tokens, 
            timeout =Config.timeout,
            embedding_model= Config.embedding_model,
            load_faiss_index = Config.load_faiss_index,
            documents_path =Config.documents_path ,
            top_k =Config.top_k ,
            system_prompt=Config.system_prompt,
        )
        return Create_Embeddings_Config
    


        



In [33]:
import pandas as pd 
from sentence_transformers import SentenceTransformer
import faiss
import pickle
from ollama import Client

In [ ]:

class CreateEmbedings:

    def __init__(
        self,
        config: CreateEmbendingsConfig
    ):

        self.config = config
        
        self.model = None
        self.index = None
        self.documents = None
        self.params = Read_Yaml(PARAMS_FILE_PATH)

    def connect(self):

        try:

            print("=" * 70)
            print("Connecting to Ollama Server")
            print("=" * 70)

            print("Host  :", self.config.base_url)
            print("Model :", self.config.model)

            self.client = Client(
                host=self.config.base_url
            )

            print("Ollama Client Created Successfully")

            return self.client

        except Exception as e:

            print("Ollama Connection Error:", e)

            raise Exception(
                "Unable to create Ollama Client."
            )

    ###############################################################
    # Test Connection
    ###############################################################
    def test_connection(self):

        try:

            print("\nChecking Ollama Server...")

            models = self.client.list()

            print("Connection Successful")

            print("\nAvailable Models:")

            for model in models.models:

                print(" -", model.model)

            return True

        except Exception as e:

            print("Actual Error:")
            print(type(e))
            print(e)

            raise Exception(
                "Unable to connect to Ollama Server."
            )

    ###############################################################
    # List Models
    ###############################################################
    def list_models(self):

        try:

            models = self.client.list()

            print("=" * 70)
            print('models ', models)
            print("Available Models")
            print("=" * 70)

            for model in models.models:

                print(model.model)
            return models

        except Exception as e:

            print("Model List Error:", e)

            raise e

    ###############################################################
    # Generate Response
    ###############################################################
    def generate_response(self, prompt):

        try:

            response = self.client.chat(
                model=self.config.model,
                messages=[
                    {
                        "role": "system",
                        "content": self.config.system_prompt
                    },

                    {
                        "role": "user",
                        "content": prompt
                    }

                ],

                options={

                    "temperature":
                        self.config.temperature,

                    "top_p":
                        self.config.top_p,

                    "num_predict":
                        self.config.max_tokens

                }
            )

            answer = response.message.content

            return answer

        except Exception as e:

            print("LLM Generation Error:")
            print(e)

            raise Exception(
                "Unable to Generate Response."
            )


    def Create_Embendings(self):

        print("=" * 70)
        print("CREATING SERVICE NOW INCIDENT EMBEDDINGS")
        print("=" * 70)

        # ------------------------------------------------------
        # Load embedding model
        # ------------------------------------------------------

        print("Loading embedding model...")

        model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2"
        )

        print("Embedding model loaded.")

        # ------------------------------------------------------
        # Validate source file
        # ------------------------------------------------------

        source_path = Path(
            self.config.source_path
        )

        print("Source file:", source_path.resolve())

        if not source_path.exists():

            raise FileNotFoundError(
                f"Source CSV not found: "
                f"{source_path.resolve()}"
            )

        # ------------------------------------------------------
        # Load ServiceNow data
        # ------------------------------------------------------

        df = pd.read_csv(
            source_path
        )

        print("Data shape:", df.shape)

        # ------------------------------------------------------
        # Required columns
        # ------------------------------------------------------

        required_columns = [

            "number",
            "short_description",
            "category",
            "subcategory",
            "priority",
            "assignment_group",
            "assigned_to",
            "resolution_hours",
            "mttr",
            "sla_percentage",
            "sla_status",
            "workload_level",
            "performance_prediction",
            "confidence",
            "points",
            "badges",
            "achievement",
            "level",
            "leaderboard_rank",
            "monthly_rank"

        ]

        # ------------------------------------------------------
        # Validate columns
        # ------------------------------------------------------

        missing_columns = [

            col
            for col in required_columns
            if col not in df.columns

        ]

        if missing_columns:

            raise ValueError(
                "Missing columns for embedding: "
                f"{missing_columns}"
            )

        print(
            "All required columns are available."
        )

        # ------------------------------------------------------
        # Create incident documents
        # ------------------------------------------------------

        documents = []

        for _, row in df.iterrows():

            text = f"""
ServiceNow Incident

Incident Number:
{row['number']}

Short Description:
{row['short_description']}

Category:
{row['category']}

Sub Category:
{row['subcategory']}

Priority:
{row['priority']}

Assignment Group:
{row['assignment_group']}

Assigned To:
{row['assigned_to']}

Resolution Hours:
{row['resolution_hours']}

MTTR:
{row['mttr']}

SLA Percentage:
{row['sla_percentage']}

SLA Status:
{row['sla_status']}

Workload Level:
{row['workload_level']}

Performance Prediction:
{row['performance_prediction']}

Performance Confidence:
{row['confidence']}

Points:
{row['points']}

Badges:
{row['badges']}

Achievement:
{row['achievement']}

Level:
{row['level']}

Leaderboard Rank:
{row['leaderboard_rank']}

Monthly Rank:
{row['monthly_rank']}
"""

            documents.append(
                text.strip()
            )

        print(
            "Total documents:",
            len(documents)
        )

        # ------------------------------------------------------
        # Generate embeddings
        # ------------------------------------------------------

        print("Generating embeddings...")

        embeddings = model.encode(

            documents,

            show_progress_bar=True

        )

        print(
            "Embedding shape:",
            embeddings.shape
        )

        # ------------------------------------------------------
        # Convert to float32
        # ------------------------------------------------------

        embeddings = embeddings.astype(
            "float32"
        )

        # ------------------------------------------------------
        # Create FAISS index
        # ------------------------------------------------------

        dimension = embeddings.shape[1]

        print(
            "Embedding dimension:",
            dimension
        )

        index = faiss.IndexFlatL2(
            dimension
        )

        # ------------------------------------------------------
        # Save FAISS index
        # ------------------------------------------------------

        output_folder=self.config.root_dir
        faiss_index_path=os.path.join(output_folder, "save_faiss_index.index")
        
        faiss.write_index(

            index,

            str(faiss_index_path)

        )

        print(
            "FAISS index saved:",
            faiss_index_path
        )

        # ------------------------------------------------------
        # Save documents
        # ------------------------------------------------------


        documents_path = faiss_index_path=os.path.join(output_folder, "documents.pkl")
       
        with open(
            documents_path,
            "wb"
        ) as f:
            pickle.dump(
                documents,
                f
            )

        print(
            "Documents saved:",
            documents_path
        )

        model_path = Path("Artifacts/LLMEmbenddings/embeddings.npy")
        model.save(str(model_path))

        #np.save("Artifacts/LLMEmbeddings/embeddings.npy",embeddings)

        # model_path = "Artifacts/LLMEmbenddings/embedding_model.pkl"

        # with open(model_path, "wb") as f:
        #     pickle.dump(model, f)



    def load_embedding_model(self):
        print("=" * 70)
        print("LOADING EMBEDDING MODEL")
        print("=" * 70)

        try:
            self.model = SentenceTransformer(
                self.config.embedding_model
            )

            print(
                "Embedding Model:",
                self.config.embedding_model
            )

            print("Embedding Model Loaded Successfully")

            return self.model

        except Exception as e:

            print("Embedding Model Error:")
            print(e)

            raise Exception(
                "Unable to load embedding model."
            )
    ###############################################################
    # Load FAISS Index
    ###############################################################

    def load_faiss_index(self):

        print("=" * 70)
        print("LOADING FAISS INDEX")
        print("=" * 70)

        index_path = Path(
            self.config.faiss_index
        )

        print(
            "FAISS Index:",
            index_path.resolve()
        )

        if not index_path.exists():

            raise FileNotFoundError(
                f"FAISS index not found: "
                f"{index_path.resolve()}"
            )

        try:

            self.index = faiss.read_index(
                str(index_path)
            )

            print(
                "FAISS Index Loaded Successfully"
            )

            print(
                "Total Vectors:",
                self.index.ntotal
            )

            return self.index

        except Exception as e:

            print("FAISS Loading Error:")
            print(e)

            raise Exception(
                "Unable to load FAISS index."
            )

    ###############################################################
    # Load Documents
    ###############################################################

    def load_documents(self):

        print("=" * 70)
        print("LOADING INCIDENT DOCUMENTS")
        print("=" * 70)

        documents_path = Path(
            self.config.documents_path
        )

        print(
            "Documents:",
            documents_path.resolve()
        )

        if not documents_path.exists():

            raise FileNotFoundError(
                f"Documents file not found: "
                f"{documents_path.resolve()}"
            )

        try:

            with open(
                documents_path,
                "rb"
            ) as f:

                self.documents = pickle.load(f)

            print(
                "Documents Loaded Successfully"
            )

            print(
                "Total Documents:",
                len(self.documents)
            )

            return self.documents

        except Exception as e:

            print("Document Loading Error:")
            print(e)

            raise Exception(
                "Unable to load incident documents."
            )
    ###############################################################
    # Load Everything
    ###############################################################

    def load_resources(self):

        print("=" * 70)
        print("LOADING FAISS RESOURCES")
        print("=" * 70)

        self.load_embedding_model()

        self.load_faiss_index()

        self.load_documents()

        print("=" * 70)
        print("ALL FAISS RESOURCES LOADED")
        print("=" * 70)
    ###############################################################
    # Create Query Embedding
    ###############################################################

    def create_query_embedding(
        self,
        query
    ):

        if self.model is None:

            self.load_embedding_model()

        print(
            "\nCreating query embedding..."
        )

        query_embedding = self.model.encode(
            [query]
        )

        query_embedding = query_embedding.astype(
            "float32"
        )

        print(
            "Query embedding shape:",
            query_embedding.shape
        )

        return query_embedding
     ###############################################################
    # Search FAISS
    ###############################################################

    def search(self,query,top_k=None):

        print("=" * 70)
        print("SEARCHING RELEVANT INCIDENTS")
        print("=" * 70)
        if self.index is None:
            self.load_faiss_index()
        if self.documents is None:
            self.load_documents()
        if self.model is None:
            self.load_embedding_model()
        # ----------------------------------------------------------
        # Get top K
        # ----------------------------------------------------------

        if top_k is None:
            top_k = self.config.top_k
        # ----------------------------------------------------------
        # Make sure K doesn't exceed document count
        # ----------------------------------------------------------

        top_k = min(top_k,len(self.documents))

        # ----------------------------------------------------------
        # Create query embedding
        # ----------------------------------------------------------

        query_embedding = self.create_query_embedding( query)

        # ----------------------------------------------------------
        # Search FAISS
        # ----------------------------------------------------------

        distances, indices = self.index.search( query_embedding, top_k)

        # ----------------------------------------------------------
        # Prepare results
        # ----------------------------------------------------------

        results = []

        for rank, (index_id, distance) in enumerate(
            zip(
                indices[0],
                distances[0]
            ),
            start=1
        ):

            # FAISS can return -1 if no result
            if index_id == -1:
                continue

            document = self.documents[index_id]

            result = {

                "rank": rank,

                "index": int(index_id),

                "distance": float(distance),

                "document": document

            }

            results.append(
                result
            )

        # ----------------------------------------------------------
        # Display results
        # ----------------------------------------------------------

        print(
            "\nQuery:",
            query
        )

        print(
            "\nRetrieved Incidents:",
            len(results)
        )

        for result in results:

            print("\n" + "-" * 70)

            print(
                "Rank:",
                result["rank"]
            )

            print(
                "FAISS Index:",
                result["index"]
            )

            print(
                "Distance:",
                round(
                    result["distance"],
                    4
                )
            )

            print(
                "Incident:"
            )

            print(
                result["document"]
            )

        print("\n" + "=" * 70)

        return results





In [ ]:
try:
    config = ConfigurationManager().get_create_embendings_config()

    embendings = CreateEmbedings(config)
     # 1. Connect to Ollama
    embendings.connect()
    
        # 2. Test connection
    embendings.test_connection()
    
        # 3. List available models
    embendings.list_models()
    
        # 4. Test simple LLM response
    response = embendings.generate_response(
            "Explain SLA Compliance in ServiceNow."
        )
    embending = embendings.Create_Embendings()

    embendings.load_embedding_model()
    embendings.load_faiss_index()
    embendings.load_documents()
    embendings.load_resources()
    #embendings.create_query_embedding()


    


except Exception as e:
    raise e

Reading YAML : Config\Config.yaml
[2026-08-09 23:15:12,885: INFO: Common: YAML file loaded successfully : Config\Config.yaml]


Reading YAML : params.yaml
[2026-08-09 23:15:12,947: INFO: Common: YAML file loaded successfully : params.yaml]
[2026-08-09 23:15:12,959: INFO: Common: Created directory : Artifacts/LLMEmbenddings]
Reading YAML : params.yaml
[2026-08-09 23:15:13,003: INFO: Common: YAML file loaded successfully : params.yaml]
Connecting to Ollama Server
Host  : http://127.0.0.1:11434
Model : llama3.2
Ollama Client Created Successfully

Checking Ollama Server...
[2026-08-09 23:15:13,243: INFO: _client: HTTP Request: GET http://127.0.0.1:11434/api/tags "HTTP/1.1 200 OK"]
Connection Successful

Available Models:
 - llama3.2:latest
[2026-08-09 23:15:13,289: INFO: _client: HTTP Request: GET http://127.0.0.1:11434/api/tags "HTTP/1.1 200 OK"]
models  models=[Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 7, 10, 17, 46, 51, 314138, tzinfo=TzInfo(19800)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='g

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 730.24it/s]


[2026-08-09 23:15:44,338: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"]
[2026-08-09 23:15:44,712: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"]
[2026-08-09 23:15:44,967: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"]
[2026-08-09 23:15:45,351: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"]
[2026-08-09 23:15:45,726: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-08-09 23:15:45,754: INFO: _client: HTTP Request: HEAD h

Batches: 100%|██████████| 2/2 [00:07<00:00,  3.89s/it]

...

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1596.74it/s]


Embedding Model: Artifacts/LLMEmbenddings/embeddings.npy
Embedding Model Loaded Successfully
LOADING FAISS INDEX
FAISS Index: C:\Users\RSR\PYTHON\ServicenowGamification\Artifacts\LLMEmbenddings\faiss_index.index
FAISS Index Loaded Successfully
Total Vectors: 56
LOADING INCIDENT DOCUMENTS
Documents: C:\Users\RSR\PYTHON\ServicenowGamification\Artifacts\LLMEmbenddings\documents.pkl
Documents Loaded Successfully
Total Documents: 63
LOADING FAISS RESOURCES
LOADING EMBEDDING MODEL
[2026-08-09 23:16:01,004: INFO: model: No device provided, using cpu]
[2026-08-09 23:16:01,023: INFO: model: Loading SentenceTransformer model from Artifacts/LLMEmbenddings/embeddings.npy.]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 748.05it/s]


Embedding Model: Artifacts/LLMEmbenddings/embeddings.npy
Embedding Model Loaded Successfully
LOADING FAISS INDEX
FAISS Index: C:\Users\RSR\PYTHON\ServicenowGamification\Artifacts\LLMEmbenddings\faiss_index.index
FAISS Index Loaded Successfully
Total Vectors: 56
LOADING INCIDENT DOCUMENTS
Documents: C:\Users\RSR\PYTHON\ServicenowGamification\Artifacts\LLMEmbenddings\documents.pkl
Documents Loaded Successfully
Total Documents: 63
ALL FAISS RESOURCES LOADED


TypeError: CreateEmbedings.create_query_embedding() missing 1 required positional argument: 'query'

In [39]:
import faiss

index_path = r"C:\Users\RSR\PYTHON\ServicenowGamification\Artifacts\LLMEmbenddings\faiss_index.index"

index = faiss.read_index(index_path)

print("FAISS dimension:", index.d)
print("Total vectors:", index.ntotal)

FAISS dimension: 384
Total vectors: 56
